In [ ]:
from transformers import AutoTokenizer
import torch

model_name = "mistralai/Mistral-7B-v0.1"
# ===== Step 1: Load tokenizer =====
tokenizer_path = f"{model_name}/tokenizer" 
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
print(f"[✓] Tokenizer loaded from: {tokenizer_path}")

# ===== Step 2: Load lm_head.weight =====
space_name = "output_proj"
lm_head_path = f"{model_name}/tensors/{space_name}.pt" 
embedding_matrix = torch.load(lm_head_path, map_location="cpu")
print(f"[✓] lm_head.weight loaded: shape = {embedding_matrix.shape}")
embedding_matrix = embedding_matrix.to(torch.float32)
# embedding_matrix = embedding_matrix.cpu().to(torch.float32).numpy()
print(f"[✓] lm_head.weight loaded: shape = {embedding_matrix.shape}")

In [ ]:
import os
import json
from collections import defaultdict


def collect_runs_by_hdbscan_config(meta_dir):
    """
    Traverse meta directory and group runs by identical HDBSCAN parameter sets.

    Returns
    -------
    dict
        {
            hdbscan_param_tuple: [
                {"file": ..., "pca_dim": ..., "seed": ...},
                ...
            ]
        }
    """

    if not os.path.isdir(meta_dir):
        raise ValueError(f"meta_dir not found: {meta_dir}")

    groups = defaultdict(list)

    for fname in sorted(os.listdir(meta_dir)):

        if not fname.endswith("_meta.json"):
            continue

        meta_path = os.path.join(meta_dir, fname)

        with open(meta_path, "r", encoding="utf-8") as f:
            meta = json.load(f)

        h = meta["hdbscan"]

        key = (
            h["min_cluster_size"],
            h["min_samples"],
            h["metric"],
            h["cluster_selection_method"],
            h["cluster_selection_epsilon"],
        )

        record = {
            "file": fname,
            "pca_dim": meta["pca"]["dim"],
            "seed": meta["pca"]["seed"],
        }

        groups[key].append(record)

    # 排序方便后续处理
    for key in groups:
        groups[key] = sorted(
            groups[key],
            key=lambda x: (x["pca_dim"], x["seed"])
        )

    return dict(groups)

In [19]:
meta_dir = "comp/mistralai/Mistral-7B-v0.1/output_proj/meta_l2=True"

groups = collect_runs_by_hdbscan_config(meta_dir)
groups

{(5,
  1,
  'euclidean',
  'eom',
  0.0): [{'file': 'run_00001_meta.json',
   'pca_dim': 8,
   'seed': 0}, {'file': 'run_00005_meta.json', 'pca_dim': 8, 'seed': 1}, {'file': 'run_00009_meta.json',
   'pca_dim': 8,
   'seed': 2}, {'file': 'run_00013_meta.json',
   'pca_dim': 47,
   'seed': 0}, {'file': 'run_00017_meta.json', 'pca_dim': 47, 'seed': 1}, {'file': 'run_00021_meta.json',
   'pca_dim': 47,
   'seed': 2}, {'file': 'run_00025_meta.json',
   'pca_dim': 108,
   'seed': 0}, {'file': 'run_00029_meta.json', 'pca_dim': 108, 'seed': 1}, {'file': 'run_00033_meta.json',
   'pca_dim': 108,
   'seed': 2}, {'file': 'run_00037_meta.json',
   'pca_dim': 158,
   'seed': 0}, {'file': 'run_00041_meta.json', 'pca_dim': 158, 'seed': 1}, {'file': 'run_00045_meta.json',
   'pca_dim': 158,
   'seed': 2}, {'file': 'run_00049_meta.json',
   'pca_dim': 1111,
   'seed': 0}, {'file': 'run_00053_meta.json', 'pca_dim': 1111, 'seed': 1}, {'file': 'run_00057_meta.json',
   'pca_dim': 1111,
   'seed': 2}, {'f

In [20]:
import os
import json
import pandas as pd


def _infer_meta_json_path_from_csv(csv_path):
    """
    Infer meta json path from cluster csv path.

    Expected pattern:
        .../clusters_l2=True/run_00001_clusters.csv
    ->  .../meta_l2=True/run_00001_meta.json
    """
    csv_path = os.path.abspath(csv_path)
    csv_dir = os.path.dirname(csv_path)
    csv_name = os.path.basename(csv_path)

    parent_dir = os.path.dirname(csv_dir)
    cluster_dir_name = os.path.basename(csv_dir)

    if not cluster_dir_name.startswith("clusters_l2="):
        return None

    meta_dir_name = cluster_dir_name.replace("clusters_l2=", "meta_l2=", 1)
    meta_dir = os.path.join(parent_dir, meta_dir_name)

    if not csv_name.endswith("_clusters.csv"):
        return None

    meta_name = csv_name.replace("_clusters.csv", "_meta.json")
    meta_path = os.path.join(meta_dir, meta_name)

    return meta_path


def _print_meta_header(meta_json_path):
    """
    Print run-level metadata if meta json exists.
    """
    if meta_json_path is None or not os.path.exists(meta_json_path):
        print("[Meta] Not found.")
        return

    with open(meta_json_path, "r", encoding="utf-8") as f:
        meta = json.load(f)

    print("=" * 80)
    print(f"[META] {meta_json_path}")
    print(f"run_id: {meta.get('run_id')}")
    print(f"model_name: {meta.get('model_name')}")
    print(f"space_name: {meta.get('space_name')}")
    print(f"matrix_shape: {meta.get('matrix_shape')}")

    pca_info = meta.get("pca", {})
    print(
        "PCA | "
        f"dim={pca_info.get('dim')} | "
        f"seed={pca_info.get('seed')}"
    )

    pca_meta = pca_info.get("meta", {})
    if pca_meta:
        print(
            "PCA meta | "
            f"svd_solver={pca_meta.get('svd_solver')} | "
            f"fit_time_sec={pca_meta.get('fit_time_sec')} | "
            f"explained_variance_ratio_sum={pca_meta.get('explained_variance_ratio_sum')}"
        )

    print(f"L2 norm: {meta.get('l2_norm')}")

    hdb = meta.get("hdbscan", {})
    print(
        "HDBSCAN | "
        f"min_cluster_size={hdb.get('min_cluster_size')} | "
        f"min_samples={hdb.get('min_samples')} | "
        f"metric={hdb.get('metric')} | "
        f"cluster_selection_method={hdb.get('cluster_selection_method')} | "
        f"cluster_selection_epsilon={hdb.get('cluster_selection_epsilon')}"
    )

    stats = meta.get("stats", {})
    if stats:
        print(
            "Run stats | "
            f"n_clusters_excl_noise={stats.get('n_clusters_excl_noise')} | "
            f"n_noise={stats.get('n_noise')} | "
            f"n_total={stats.get('n_total')} | "
            f"noise_ratio={stats.get('noise_ratio')} | "
            f"avg_prob_all={stats.get('avg_prob_all')} | "
            f"avg_prob_assigned={stats.get('avg_prob_assigned')}"
        )

    outputs = meta.get("outputs", {})
    if outputs:
        print(f"cluster_csv: {outputs.get('cluster_csv')}")
        print(f"meta_json: {outputs.get('meta_json')}")

    print("=" * 80)


def print_cluster_result_from_csv(
    csv_path,
    tokenizer,
    top_k=20,
    include_noise_tokens=False,
    sort_clusters_by="cluster_id",   # "cluster_id" | "size" | "mean_prob"
):
    """
    Read one token-level cluster CSV from the PCA -> L2 -> GPU HDBSCAN pipeline
    and print cluster summaries.

    It will also automatically infer and print the matching meta JSON if found.

    Expected CSV columns:
        - token_id
        - cluster_id
        - probability

    Args:
        csv_path: path to clusters_l2=.../run_XXXXX_clusters.csv
        tokenizer: HF tokenizer
        top_k: print top-k highest-probability tokens per cluster
        include_noise_tokens: whether to print all noise tokens
        sort_clusters_by:
            - "cluster_id": ascending cluster id
            - "size": descending cluster size
            - "mean_prob": descending cluster mean probability
    """
    # ===== print inferred meta first =====
    meta_json_path = _infer_meta_json_path_from_csv(csv_path)
    _print_meta_header(meta_json_path)

    df = pd.read_csv(csv_path)

    required_cols = {"token_id", "cluster_id", "probability"}
    if not required_cols.issubset(df.columns):
        raise ValueError(
            f"CSV must contain columns {required_cols}, got {set(df.columns)}"
        )

    total_tokens = len(df)

    # ===== noise statistics =====
    noise_df = df[df["cluster_id"] == -1].copy()
    noise_tokens = len(noise_df)
    noise_ratio = (noise_tokens / total_tokens * 100.0) if total_tokens > 0 else 0.0

    # ===== non-noise statistics =====
    df_no_noise = df[df["cluster_id"] != -1].copy()

    if len(df_no_noise) == 0:
        print(f"[File] {csv_path}")
        print("No non-noise clusters found.")
        print(
            f"Noise tokens: {noise_tokens} / {total_tokens} "
            f"({noise_ratio:.2f}%)"
        )
        if include_noise_tokens and noise_tokens > 0:
            print("\nNoise tokens (cluster_id = -1):")
            noise_token_strs = []
            for tid in noise_df["token_id"].astype(int).tolist():
                try:
                    tok_str = tokenizer.decode([tid])
                except Exception:
                    tok_str = tokenizer.convert_ids_to_tokens([tid])[0]
                noise_token_strs.append(tok_str)
            print(", ".join(noise_token_strs))
        return

    global_mean = df_no_noise["probability"].mean()
    global_std = df_no_noise["probability"].std(ddof=0)

    # ===== cluster-level summary table =====
    cluster_summary = (
        df_no_noise.groupby("cluster_id", as_index=False)
        .agg(
            size=("token_id", "count"),
            mean_prob=("probability", "mean"),
            std_prob=("probability", "std"),
            max_prob=("probability", "max"),
            min_prob=("probability", "min"),
        )
    )

    cluster_summary["std_prob"] = cluster_summary["std_prob"].fillna(0.0)

    if sort_clusters_by == "cluster_id":
        cluster_summary = cluster_summary.sort_values("cluster_id", ascending=True)
    elif sort_clusters_by == "size":
        cluster_summary = cluster_summary.sort_values(
            ["size", "cluster_id"], ascending=[False, True]
        )
    elif sort_clusters_by == "mean_prob":
        cluster_summary = cluster_summary.sort_values(
            ["mean_prob", "cluster_id"], ascending=[False, True]
        )
    else:
        raise ValueError(
            "sort_clusters_by must be one of: 'cluster_id', 'size', 'mean_prob'"
        )

    n_clusters = len(cluster_summary)

    print(f"[File] {csv_path}")
    print(
        f"Clusters (excluding noise): {n_clusters}\n"
        f"Total tokens: {total_tokens}\n"
        f"Noise tokens: {noise_tokens} / {total_tokens} ({noise_ratio:.2f}%)\n"
        f"Global probability mean = {global_mean:.4f}, "
        f"std = {global_std:.4f} (excluding noise)"
    )
    print("-" * 80)

    # ===== print each cluster =====
    for _, row in cluster_summary.iterrows():
        cid = int(row["cluster_id"])
        size = int(row["size"])
        mean_prob = float(row["mean_prob"])
        std_prob = float(row["std_prob"])
        max_prob = float(row["max_prob"])
        min_prob = float(row["min_prob"])

        sub = df_no_noise[df_no_noise["cluster_id"] == cid].copy()
        top = sub.sort_values("probability", ascending=False).head(top_k)

        tokens = []
        for tid in top["token_id"].astype(int).tolist():
            try:
                tok_str = tokenizer.decode([tid])
            except Exception:
                tok_str = tokenizer.convert_ids_to_tokens([tid])[0]
            tokens.append(tok_str)

        print(
            f"Cluster {cid} | "
            f"size={size} | "
            f"mean_prob={mean_prob:.4f} | "
            f"std_prob={std_prob:.4f} | "
            f"max_prob={max_prob:.4f} | "
            f"min_prob={min_prob:.4f}"
        )
        print(", ".join(tokens))
        print()

    # ===== optionally print noise tokens =====
    if include_noise_tokens:
        if noise_tokens > 0:
            print("-" * 80)
            print("Noise tokens (cluster_id = -1):")
            noise_token_strs = []
            for tid in noise_df["token_id"].astype(int).tolist():
                try:
                    tok_str = tokenizer.decode([tid])
                except Exception:
                    tok_str = tokenizer.convert_ids_to_tokens([tid])[0]
                noise_token_strs.append(tok_str)
            print(", ".join(noise_token_strs))
        else:
            print("-" * 80)
            print("No noise tokens.")

In [27]:
import os
from contextlib import redirect_stdout

def print_hdbscan_group_by_params(
    groups,
    meta_dir,
    tokenizer,
    min_cluster_size,
    min_samples,
    metric,
    cluster_selection_method,
    cluster_selection_epsilon,
    seed=None,
    top_k=20,
):
    """
    Print cluster results for runs with the specified HDBSCAN parameters.

    seed:
        None  -> print all seeds
        int   -> print only this seed
    """

    key = (
        min_cluster_size,
        min_samples,
        metric,
        cluster_selection_method,
        cluster_selection_epsilon,
    )

    if key not in groups:
        raise ValueError(f"HDBSCAN config not found: {key}")

    runs = groups[key]

    # 如果指定 seed → 过滤
    if seed is not None:
        runs = [r for r in runs if r["seed"] == seed]

        if len(runs) == 0:
            raise ValueError(f"No runs found with seed={seed}")

    # 推断 clusters 目录
    parent_dir = os.path.dirname(meta_dir)
    meta_name = os.path.basename(meta_dir)

    clusters_dir = meta_name.replace("meta_l2=", "clusters_l2=")
    clusters_dir = os.path.join(parent_dir, clusters_dir)

    parent_dir = os.path.dirname(meta_dir)
    meta_name = os.path.basename(meta_dir)   # meta_l2=True
    l2_suffix = meta_name.replace("meta_l2=", "")
    reader_dir = os.path.join(parent_dir, f"cluster_reader_l2={l2_suffix}")
    os.makedirs(reader_dir, exist_ok=True)
    
    output_file = (
        f"hdbscan_mcs{min_cluster_size}"
        f"_ms{min_samples}"
        f"_metric{metric}"
        f"_method{cluster_selection_method}"
        f"_eps{cluster_selection_epsilon}"
    )
    
    if seed is not None:
        output_file += f"_seed{seed}"
    
    output_file += ".txt"

    output_file = os.path.join(reader_dir, output_file)

    with open(output_file, "w", encoding="utf-8") as f, redirect_stdout(f):

        print("\nSelected HDBSCAN config:")
        print(key)
        print(f"seed filter: {seed}")
        print("=" * 80)
    
        for r in runs:
    
            run_name = r["file"].replace("_meta.json", "")
            csv_name = run_name + "_clusters.csv"
    
            csv_path = os.path.join(clusters_dir, csv_name)
    
            if not os.path.exists(csv_path):
                print(f"[WARN] missing file: {csv_path}")
                continue
    
            print("\n" + "#" * 80)
            print(f"RUN FILE: {r['file']}")
            print(f"PCA dim={r['pca_dim']}  seed={r['seed']}")
            print("#" * 80)
    
            print_cluster_result_from_csv(
                csv_path=csv_path,
                tokenizer=tokenizer,
                top_k=top_k,
            )

In [30]:
print_hdbscan_group_by_params(
    groups,
    meta_dir,
    tokenizer,
    min_cluster_size=5,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
    cluster_selection_epsilon=0.0,
    seed=2
)

In [5]:
csv_path = "comp/mistralai/Mistral-7B-v0.1/output_proj/clusters_l2=True/run_00027_clusters.csv"

print_cluster_result_from_csv(
    csv_path=csv_path,
    tokenizer=tokenizer,
    top_k=20,
    include_noise_tokens=False,
    sort_clusters_by="size",
)

[META] /home/void/Projects/EmbdAlys/comp/mistralai/Mistral-7B-v0.1/output_proj/meta_l2=True/run_00027_meta.json
run_id: 27
model_name: mistralai/Mistral-7B-v0.1
space_name: output_proj
matrix_shape: [32000, 4096]
PCA | dim=108 | seed=0
PCA meta | svd_solver=randomized | fit_time_sec=2.0034573078155518 | explained_variance_ratio_sum=0.09379968792200089
L2 norm: True
HDBSCAN | min_cluster_size=10 | min_samples=1 | metric=euclidean | cluster_selection_method=eom | cluster_selection_epsilon=0.0
Run stats | n_clusters_excl_noise=365 | n_noise=24270 | n_total=32000 | noise_ratio=0.7584375 | avg_prob_all=0.2325766384601593 | avg_prob_assigned=0.9628010988235474
cluster_csv: comp/mistralai/Mistral-7B-v0.1/output_proj/clusters/run_00027_clusters.csv
meta_json: comp/mistralai/Mistral-7B-v0.1/output_proj/meta/run_00027_meta.json
[File] comp/mistralai/Mistral-7B-v0.1/output_proj/clusters_l2=True/run_00027_clusters.csv
Clusters (excluding noise): 365
Total tokens: 32000
Noise tokens: 24270 / 32000 